# Day 12 — boto3: S3 Operations

**Phase 1 · Module 2: Knowledge Bases** · ~30 min

Yesterday you *called* a model. Today you learn where a RAG system's **documents live**: **Amazon S3** — AWS's object store. Tomorrow's Bedrock Knowledge Base reads straight from an S3 bucket, so every upload, download, and access rule you practise here is the *data layer* under your agent's memory. Four moves: **put an object, get it back (streamed), share it with a signed URL, and upload a big file in parts.**

> **Keyless & offline.** A real `boto3.client("s3")` needs AWS credentials. We use a tiny **`FakeS3`** with the same method names and return shapes (a dict with a streaming `Body`, etc.). Swap in `boto3.client("s3")` later and your calling code is unchanged.


## 1. Buckets & keys — the mental model

S3 has no folders. It has **buckets** (a namespace) holding **objects**, each addressed by a **key** — a string that *looks* like a path. `kb-docs/policies/aml.pdf` is one flat key; the slashes are just characters. That 'prefix' convention is how a Knowledge Base groups a data source.

```python
import boto3
s3 = boto3.client("s3", region_name="eu-west-2")
```


In [1]:
import io, hashlib, time

class ClientError(Exception):
    def __init__(self, code, msg):
        self.response={"Error":{"Code":code,"Message":msg}}; super().__init__(f"{code}: {msg}")

class _Body:
    """Mimics a boto3 StreamingBody: .read() and .iter_chunks()."""
    def __init__(self, data): self._data=data
    def read(self): return self._data
    def iter_chunks(self, chunk_size=1024):
        for i in range(0, len(self._data), chunk_size):
            yield self._data[i:i+chunk_size]

class FakeS3:
    """Offline stand-in for boto3.client('s3'). Same methods + response shapes."""
    def __init__(self): self.store={}   # (bucket,key) -> dict(body,meta,content_type)
    def put_object(self, Bucket, Key, Body, ContentType="binary/octet-stream", Metadata=None):
        data = Body.encode() if isinstance(Body, str) else Body
        self.store[(Bucket,Key)] = {"body":data, "ContentType":ContentType, "Metadata":Metadata or {}}
        return {"ETag": '"'+hashlib.md5(data).hexdigest()+'"'}
    def get_object(self, Bucket, Key):
        if (Bucket,Key) not in self.store:
            raise ClientError("NoSuchKey", f"key {Key} not found")
        o=self.store[(Bucket,Key)]
        return {"Body": _Body(o["body"]), "ContentType": o["ContentType"],
                "Metadata": o["Metadata"], "ContentLength": len(o["body"])}
    def list_objects_v2(self, Bucket, Prefix=""):
        keys=[{"Key":k, "Size":len(v["body"])} for (b,k),v in self.store.items()
              if b==Bucket and k.startswith(Prefix)]
        return {"Contents": sorted(keys, key=lambda x:x["Key"]), "KeyCount": len(keys)}
    def generate_presigned_url(self, op, Params, ExpiresIn=3600):
        b,k=Params["Bucket"],Params["Key"]
        sig=hashlib.sha1(f"{b}/{k}/{ExpiresIn}".encode()).hexdigest()[:16]
        return f"https://{b}.s3.eu-west-2.amazonaws.com/{k}?X-Amz-Expires={ExpiresIn}&X-Amz-Signature={sig}"
    def upload_file(self, Filename, Bucket, Key, Config=None):
        data=Filename.encode() if isinstance(Filename,str) else Filename
        parts = 1 if Config is None else max(1, len(data)//Config.multipart_chunksize + 1)
        self.store[(Bucket,Key)]={"body":data,"ContentType":"application/pdf","Metadata":{}}
        return parts

s3 = FakeS3()
BUCKET = "barclays-kb-docs"
print("s3 client ready ·", BUCKET)


s3 client ready · barclays-kb-docs


## 2. `put_object` — upload with **ContentType** + **metadata**

`put_object` writes bytes to a key. Two arguments matter for a Knowledge Base: **`ContentType`** (so downstream tools know it's `application/pdf` vs `text/plain`) and **`Metadata`** (your own key/values — e.g. which department, retention class). Metadata rides *with* the object.


In [2]:
doc = "AML POLICY: report any cash transaction over 10000 GBP within 24 hours."

resp = s3.put_object(
    Bucket=BUCKET,
    Key="policies/aml.txt",
    Body=doc,
    ContentType="text/plain",
    Metadata={"department": "compliance", "retention": "7y"},
)
print("stored. ETag:", resp["ETag"])


stored. ETag: "62b75d380798a546845a1f2d7c06a375"


☝ `Metadata` keys become `x-amz-meta-*` headers on the object — a cheap way to tag documents so a retrieval step (or an audit) can filter by `department` without opening the file. Always set `ContentType`; a missing one defaults to `binary/octet-stream` and confuses parsers.


## 3. `get_object` — download and **stream** with `iter_chunks()`

`get_object` returns a dict whose `Body` is a **streaming** object, *not* bytes. You can `.read()` it all, but for a large file you **iterate chunks** so you never hold the whole thing in memory — the same discipline as reading a big PDF into a chunker.


In [3]:
obj = s3.get_object(Bucket=BUCKET, Key="policies/aml.txt")
print("content-type:", obj["ContentType"], "| bytes:", obj["ContentLength"])
print("metadata    :", obj["Metadata"])

# stream it in small pieces instead of one big .read()
received = b""
for chunk in obj["Body"].iter_chunks(chunk_size=16):
    received += chunk
print("reassembled :", received.decode())


content-type: text/plain | bytes: 71
metadata    : {'department': 'compliance', 'retention': '7y'}
reassembled : AML POLICY: report any cash transaction over 10000 GBP within 24 hours.


☝ `iter_chunks()` hands you bytes a slice at a time — memory stays flat whether the object is 16 B or 16 GB. For a one-liner you'd write `obj["Body"].read().decode()`; reach for streaming the moment files get big (scanned invoices, PDF statements).


## 4. `generate_presigned_url` — share **without** giving away keys

A bucket is private. To let a browser, a customer, or another service fetch one object **temporarily** you mint a **pre-signed URL**: a normal HTTPS link that carries a time-limited signature. It expires (here in 3600 s = 1 hour) and needs no AWS credentials from whoever clicks it.


In [4]:
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": "policies/aml.txt"},
    ExpiresIn=3600,
)
print(url)


https://barclays-kb-docs.s3.eu-west-2.amazonaws.com/policies/aml.txt?X-Amz-Expires=3600&X-Amz-Signature=0e7b4b5320d44961


☝ Anyone with this link can read *that one object* until it expires — then it 403s. This is how you surface a source document behind a RAG citation (‘view the policy this answer came from’) without making the bucket public. Keep `ExpiresIn` short.


## 5. Large PDFs — **multipart** upload with a Transfer Manager

`put_object` is one HTTP request — fine up to a point, but a 500 MB statement archive should go up in **parts** (parallel, resumable on failure). boto3's `upload_file` does this automatically; a `TransferConfig` sets the part size and the threshold at which multipart kicks in.


In [5]:
class TransferConfig:
    def __init__(self, multipart_threshold=8*1024*1024, multipart_chunksize=8*1024*1024):
        self.multipart_threshold=multipart_threshold; self.multipart_chunksize=multipart_chunksize

big_pdf = b"%PDF-1.7 ... " + b"x"*50   # pretend this is a large file's bytes
cfg = TransferConfig(multipart_chunksize=16)   # tiny chunks so we can see splitting

parts = s3.upload_file(big_pdf, BUCKET, "statements/2026/archive.pdf", Config=cfg)
print(f"uploaded in {parts} part(s)")
# real boto3:  s3.upload_file('archive.pdf', BUCKET, 'statements/2026/archive.pdf', Config=cfg)


uploaded in 4 part(s)


☝ `upload_file` (not `put_object`) is the one you want for files: it handles multipart, retries a failed part instead of the whole upload, and parallelises. The `TransferConfig` is where you tune part size. For a Knowledge Base you'd bulk-load a corpus this way.


## 6. Organise by **prefix** — the KB data-source layout

A Bedrock Knowledge Base points at a bucket **prefix** and ingests everything under it. So the way you *lay out* keys **is** your data-source design. `list_objects_v2(Prefix=...)` is how you (and the KB) enumerate a source.


In [6]:
# add a couple more docs so there's something to list
s3.put_object(Bucket=BUCKET, Key="policies/kyc.txt", Body="KYC: verify identity before onboarding.", ContentType="text/plain")
s3.put_object(Bucket=BUCKET, Key="faqs/cards.txt",   Body="Lost card: freeze in-app, reissue in 3 days.", ContentType="text/plain")

listing = s3.list_objects_v2(Bucket=BUCKET, Prefix="policies/")
print(f"objects under 'policies/': {listing['KeyCount']}")
for o in listing["Contents"]:
    print(f"  {o['Key']:24} {o['Size']:>4} B")


objects under 'policies/': 2
  policies/aml.txt           71 B
  policies/kyc.txt           39 B


☝ Point tomorrow's Knowledge Base at `s3://barclays-kb-docs/policies/` and it ingests exactly these two files — change the prefix, change the corpus. Clean key layout = clean, filterable RAG sources.


## 7. Recap + checklist

| Task | boto3 call | Watch out for |
|---|---|---|
| **upload bytes** | `put_object(Bucket,Key,Body,ContentType,Metadata)` | always set `ContentType` |
| **download** | `get_object(...)["Body"]` | it's a **stream**, not bytes |
| **stream big** | `Body.iter_chunks(chunk_size)` | keeps memory flat |
| **share safely** | `generate_presigned_url(op,Params,ExpiresIn)` | keep expiry short |
| **big files** | `upload_file(..., Config=TransferConfig)` | use over `put_object` for files |
| **enumerate** | `list_objects_v2(Bucket,Prefix)` | prefix = your KB data source |

**Your 30 min today**
- [ ] `put_object` with a `ContentType` and one `Metadata` tag.
- [ ] `get_object` and consume `Body.iter_chunks()`.
- [ ] Mint a pre-signed URL and read its `X-Amz-Expires`.
- [ ] Say why `upload_file` beats `put_object` for a 500 MB PDF.

